# Proyecto Final — Ciencia de Datos I

## Abstract

Los personajes del videojuego *The Sims 4* —los "Sims"— poseen un sistema de bienestar emocional determinado por cientos de variables: su carrera, sus relaciones sociales, sus rasgos de personalidad, sus actividades de ocio y su estado de ánimo general. Este dataset sintético replica esa lógica y reúne información de **12 000 Sims** con **23 variables** que describen su situación vital.

El **objetivo** de este proyecto es desarrollar un modelo de clasificación binaria capaz de predecir si un Sim es **feliz** (`feliz = 1`) o **no es feliz** (`feliz = 0`), entendiendo como umbral un `animo_score ≥ 70` sobre una escala de 0 a 100.

En la primera entrega se abordó la fase exploratoria (EDA) mediante auditoría de calidad, estadística descriptiva y visualizaciones con Pandas, Matplotlib y Seaborn. En esta entrega final se avanza hacia el modelado: luego de dividir el dataset y codificar correctamente las variables categóricas mediante *One-Hot Encoding* (ajustado exclusivamente sobre el conjunto de entrenamiento), se entrena un **Árbol de Decisión** para clasificar la felicidad. La evaluación se realiza sobre ambos conjuntos para detectar sobreajuste, y el análisis de importancia de variables revela cuáles son los factores más influyentes.

## Diccionario de Datos

| Variable | Tipo | Descripción |
|---|---|---|
| **nivel_carrera** | Numérica (discreta) | Nivel alcanzado en la carrera laboral (1–10) |
| **simoleons** | Numérica (continua) | Dinero acumulado del Sim (100–100 000) |
| **nivel_habilidad** | Numérica (discreta) | Habilidad principal desarrollada (0–10) |
| **num_amigos** | Numérica (discreta) | Cantidad de amigos activos (0–20) |
| **tiene_pareja** | Booleana | Si el Sim tiene pareja romántica (1 = sí, 0 = no) |
| **num_mascotas** | Numérica (discreta) | Cantidad de mascotas (0–4) |
| **horas_ocio** | Numérica (continua) | Horas diarias de ocio (0–14) |
| **num_hijos** | Numérica (discreta) | Cantidad de hijos (0–5) |
| **nivel_social** | Numérica (discreta) | Nivel de vida social (1–10) |
| **horas_trabajo** | Numérica (continua) | Horas diarias de trabajo (0–12) |
| **nivel_fitness** | Numérica (discreta) | Condición física (0–10) |
| **tiene_enemigos** | Booleana | Si el Sim tiene enemigos declarados (1 = sí, 0 = no) |
| **moodlets_positivos** | Numérica (discreta) | Cantidad de moodlets positivos activos (0–10) |
| **moodlets_negativos** | Numérica (discreta) | Cantidad de moodlets negativos activos (0–10) |
| **num_logros** | Numérica (discreta) | Logros desbloqueados en el juego (0–50) |
| **aspiracion** | Categórica (nominal) | Aspiración de vida del Sim (8 categorías) |
| **rasgo** | Categórica (nominal) | Rasgo de personalidad dominante (8 categorías) |
| **carrera** | Categórica (nominal) | Carrera laboral activa (8 categorías) |
| **etapa_vida** | Categórica (nominal) | Etapa vital (Niño, Adolescente, Adulto Joven, Adulto, Adulto Mayor) |
| **tipo_hogar** | Categórica (nominal) | Tipo de vivienda (4 categorías) |
| **mundo** | Categórica (nominal) | Vecindario donde vive el Sim (6 categorías) |
| **animo_score** | Numérica (continua) | Puntaje de estado de ánimo general (0–100). **Se elimina antes del modelado** para evitar target leakage, ya que fue la variable usada para construir el target `feliz`. |
| **feliz** | Binaria (target) | Variable objetivo: 1 si `animo_score ≥ 70`, 0 si no |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
df = pd.read_csv("sims4_dataset.csv")
df.head()

In [ ]:
print(f"Dimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\nColumnas: {list(df.columns)}")

In [ ]:
df.info()

In [ ]:
print("Valores nulos por columna:")
print(df.isnull().sum())
print(f"\nTotal de valores nulos: {df.isnull().sum().sum()}")
print(f"Filas completamente duplicadas: {df.duplicated().sum()}")

In [ ]:
df.describe()

In [ ]:
conteo_target = df['feliz'].value_counts()
porcentaje_target = df['feliz'].value_counts(normalize=True) * 100

tabla_target = pd.DataFrame({
    'Cantidad': conteo_target,
    'Porcentaje (%)': porcentaje_target.round(2)
}).rename(index={1: 'Feliz (1)', 0: 'No Feliz (0)'})
print(tabla_target)

In [ ]:
plt.figure(figsize=(6, 6))
plt.pie(
    conteo_target,
    labels=['Feliz', 'No Feliz'],
    autopct='%1.1f%%',
    startangle=140,
    colors=['#66b3ff', '#ff9999']
)
plt.title('Distribución de la Variable Objetivo — Sims Felices vs No Felices',
          fontsize=13, pad=20)
plt.show()

## Hipótesis 1

**¿Los Sims con más amigos son significativamente más felices?**

La vida social es un pilar del bienestar en The Sims 4. Se espera que un mayor número de amigos se asocie con una mayor proporción de Sims felices.

Se segmenta `num_amigos` en cuatro rangos y se analiza la tasa de felicidad en cada grupo.

In [ ]:
df['rango_amigos'] = pd.cut(
    df['num_amigos'],
    bins=[-1, 4, 9, 14, 20],
    labels=['Pocos (0–4)', 'Algunos (5–9)', 'Varios (10–14)', 'Muchos (15–20)']
)

tasa_h1 = (df.groupby('rango_amigos', observed=True)['feliz']
             .mean()
             .mul(100)
             .round(2)
             .reset_index())
tasa_h1.columns = ['Rango de amigos', 'Tasa de felicidad (%)']
print(tasa_h1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot animo_score vs rango
sns.boxplot(
    data=df, x='rango_amigos', y='animo_score',
    order=['Pocos (0–4)', 'Algunos (5–9)', 'Varios (10–14)', 'Muchos (15–20)'],
    palette='Blues', ax=axes[0]
)
axes[0].set_title('Distribución de Ánimo por Cantidad de Amigos', fontsize=11)
axes[0].set_xlabel('Rango de Amigos')
axes[0].set_ylabel('Puntaje de Ánimo (animo_score)')

# Barras tasa felicidad
colores_h1 = ['#aec6e8', '#5fa8d3', '#2176ae', '#0a3d62']
sns.barplot(
    data=tasa_h1, x='Rango de amigos', y='Tasa de felicidad (%)',
    palette=colores_h1, hue='Rango de amigos', width=0.5, ax=axes[1]
)
axes[1].set_title('Tasa de Felicidad por Cantidad de Amigos', fontsize=11)
axes[1].set_xlabel('Rango de Amigos')
axes[1].set_ylabel('Porcentaje de Sims felices (%)')
axes[1].set_ylim(0, 100)
if axes[1].legend_:
    axes[1].legend_.remove()
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.1f%%', padding=3)

plt.suptitle('Hipótesis 1: Vida Social y Felicidad', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Conclusión Hipótesis 1:** La hipótesis se **confirma**.

La tasa de felicidad crece de forma consistente con la cantidad de amigos: los Sims con muchos amigos (15–20) son felices en una proporción notablemente mayor que los Sims con pocos amigos (0–4). El boxplot refuerza el patrón: la mediana de `animo_score` sube progresivamente en cada grupo. Esto valida que `num_amigos` es una variable relevante para el modelo de clasificación.

## Hipótesis 2

**¿Los Sims con mayor nivel de carrera presentan mayor felicidad?**

Un Sim exitoso laboralmente accede a mejores ingresos, mayor autoestima y más moodlets positivos. Se espera que a mayor `nivel_carrera`, mayor sea la tasa de felicidad.

In [ ]:
df['grupo_carrera'] = pd.cut(
    df['nivel_carrera'],
    bins=[0, 3, 6, 10],
    labels=['Inicio (1–3)', 'Medio (4–6)', 'Alto (7–10)']
)

tasa_h2 = (df.groupby('grupo_carrera', observed=True)['feliz']
             .mean()
             .mul(100)
             .round(2)
             .reset_index())
tasa_h2.columns = ['Nivel de carrera', 'Tasa de felicidad (%)']
print(tasa_h2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(
    data=df, x='grupo_carrera', y='animo_score',
    order=['Inicio (1–3)', 'Medio (4–6)', 'Alto (7–10)'],
    palette=['#f7b2ad', '#f4845f', '#e63946'], ax=axes[0]
)
axes[0].set_title('Distribución de Ánimo por Nivel de Carrera', fontsize=11)
axes[0].set_xlabel('Nivel de Carrera')
axes[0].set_ylabel('Puntaje de Ánimo (animo_score)')

colores_h2 = ['#f7b2ad', '#f4845f', '#e63946']
sns.barplot(
    data=tasa_h2, x='Nivel de carrera', y='Tasa de felicidad (%)',
    palette=colores_h2, hue='Nivel de carrera', width=0.5, ax=axes[1]
)
axes[1].set_title('Tasa de Felicidad por Nivel de Carrera', fontsize=11)
axes[1].set_xlabel('Nivel de Carrera')
axes[1].set_ylabel('Porcentaje de Sims felices (%)')
axes[1].set_ylim(0, 100)
if axes[1].legend_:
    axes[1].legend_.remove()
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.1f%%', padding=3)

plt.suptitle('Hipótesis 2: Carrera Laboral y Felicidad', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Conclusión Hipótesis 2:** La hipótesis se **confirma**.

Los Sims con carrera en nivel alto (7–10) presentan una tasa de felicidad significativamente mayor que los de nivel inicial (1–3). El progreso laboral actúa como motor del bienestar en el juego, lo que justifica incluir `nivel_carrera` entre las variables del modelo.

In [ ]:
vars_numericas = ['nivel_carrera', 'simoleons', 'nivel_habilidad', 'num_amigos',
                  'tiene_pareja', 'num_mascotas', 'horas_ocio', 'num_hijos',
                  'nivel_social', 'horas_trabajo', 'nivel_fitness', 'tiene_enemigos',
                  'moodlets_positivos', 'moodlets_negativos', 'num_logros']

matriz_corr = df[vars_numericas].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    matriz_corr, annot=True, cmap='coolwarm', fmt='.2f',
    linewidths=0.5, vmin=-1, vmax=1
)
plt.title('Mapa de Calor — Correlación entre Variables Numéricas', fontsize=13, pad=20)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

Las correlaciones entre las variables numéricas son en general bajas, lo que indica que no existe una relación lineal fuerte entre ellas. Esto valida la elección de un modelo basado en árboles de decisión, que puede capturar relaciones no lineales y combinaciones condicionales entre variables.

## División de Conjuntos y Codificación de Variables

### Eliminación de variables no utilizables en el modelo

- **`animo_score`** se elimina porque fue la variable usada para *construir* el target `feliz`. Incluirla generaría **target leakage** inmediato.
- Las columnas auxiliares de agrupación (`rango_amigos`, `grupo_carrera`) creadas solo para el EDA también se descartan.

### Identificación de variables categóricas

Las variables nominales (`aspiracion`, `rasgo`, `carrera`, `etapa_vida`, `tipo_hogar`, `mundo`) serán codificadas con **OneHotEncoder**, ajustado *exclusivamente sobre el conjunto de entrenamiento*, y luego aplicado al conjunto de prueba sin reajuste, para evitar **data leakage**.

In [ ]:
COLS_ELIMINAR = ['animo_score', 'rango_amigos', 'grupo_carrera']
X = df.drop(columns=['feliz'] + COLS_ELIMINAR, errors='ignore')
y = df['feliz']

print(f"Variables de entrada (X): {X.shape[1]} columnas")
print(f"Variable objetivo (y): {y.shape[0]} filas")
print(f"\nBalance de clases:")
print(y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape[0]} filas  |  X_test: {X_test.shape[0]} filas")
print(f"\nBalance en TRAIN:")
print(y_train.value_counts())
print(f"\nBalance en TEST:")
print(y_test.value_counts())

In [ ]:
COLS_NOMINALES = ['aspiracion', 'rasgo', 'carrera', 'etapa_vida', 'tipo_hogar', 'mundo']
COLS_NUMERICAS = [c for c in X_train.columns if c not in COLS_NOMINALES]

print(f"Variables nominales ({len(COLS_NOMINALES)}): {COLS_NOMINALES}")
print(f"Variables numéricas ({len(COLS_NUMERICAS)}): {COLS_NUMERICAS}")

In [ ]:
# El encoder se AJUSTA solo con X_train y luego se APLICA a ambos
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
ohe.fit(X_train[COLS_NOMINALES])

ohe_cols = ohe.get_feature_names_out(COLS_NOMINALES)

X_train_cat = pd.DataFrame(
    ohe.transform(X_train[COLS_NOMINALES]),
    columns=ohe_cols,
    index=X_train.index
)
X_test_cat = pd.DataFrame(
    ohe.transform(X_test[COLS_NOMINALES]),
    columns=ohe_cols,
    index=X_test.index
)

print(f"Nuevas columnas OHE generadas: {len(ohe_cols)}")
print(list(ohe_cols[:10]), "...")

In [ ]:
X_train_final = pd.concat([X_train[COLS_NUMERICAS], X_train_cat], axis=1)
X_test_final  = pd.concat([X_test[COLS_NUMERICAS],  X_test_cat],  axis=1)

print(f"X_train_final: {X_train_final.shape[0]} filas × {X_train_final.shape[1]} columnas")
print(f"X_test_final:  {X_test_final.shape[0]}  filas × {X_test_final.shape[1]} columnas")
X_train_final.head()

In [ ]:
modelo = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=30,
    class_weight='balanced',
    random_state=42
)
modelo.fit(X_train_final, y_train)
print("Modelo entrenado correctamente.")

In [ ]:
y_pred_train = modelo.predict(X_train_final)
y_pred_test  = modelo.predict(X_test_final)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test  = accuracy_score(y_test,  y_pred_test)

print(f"Accuracy TRAIN : {acc_train:.4f} ({acc_train*100:.2f} %)")
print(f"Accuracy TEST  : {acc_test:.4f}  ({acc_test*100:.2f} %)")
print(f"Diferencia     : {abs(acc_train - acc_test)*100:.2f} pp")

In [ ]:
print("=" * 50)
print("REPORTE DE CLASIFICACIÓN — CONJUNTO DE ENTRENAMIENTO")
print("=" * 50)
print(classification_report(y_train, y_pred_train,
                             target_names=['No Feliz (0)', 'Feliz (1)']))

print("=" * 50)
print("REPORTE DE CLASIFICACIÓN — CONJUNTO DE PRUEBA")
print("=" * 50)
print(classification_report(y_test, y_pred_test,
                             target_names=['No Feliz (0)', 'Feliz (1)']))

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicho: No Feliz (0)', 'Predicho: Feliz (1)'],
    yticklabels=['Real: No Feliz (0)', 'Real: Feliz (1)']
)
plt.title('Matriz de Confusión — Conjunto de Prueba', fontsize=13, pad=15)
plt.xlabel('Predicción del modelo')
plt.ylabel('Valor real')
plt.tight_layout()
plt.show()

## Evaluación del Sobreajuste

Al comparar el rendimiento entre el conjunto de entrenamiento y el de prueba, se puede diagnosticar si el modelo generalizó bien o si sobreajustó los datos con los que aprendió.

- Si la diferencia entre accuracy de train y test es **pequeña (< 5 pp)**: el modelo generalizó correctamente.
- Si la diferencia es **grande (> 10 pp)**: hay sobreajuste; el modelo memorizó patrones del entrenamiento que no se replican en datos nuevos.

Los parámetros `max_depth=6` y `min_samples_leaf=30` limitan la complejidad del árbol para reducir el riesgo de sobreajuste sin sacrificar capacidad predictiva.

In [ ]:
importancias = pd.Series(
    modelo.feature_importances_,
    index=X_train_final.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importancias.head(15).sort_values().plot(kind='barh', color='#2176ae')
plt.title('Top 15 Variables más Importantes — Árbol de Decisión', fontsize=13, pad=15)
plt.xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

print("\nTop 10 variables:")
print(importancias.head(10).to_string())

## Conclusión General

Este proyecto analizó los factores que determinan la felicidad de los Sims en The Sims 4 usando un dataset sintético de **12 000 Sims con 23 variables**.

El análisis exploratorio confirmó las dos hipótesis planteadas:
1. Los Sims con más amigos son proporcionalmente más felices.
2. Los Sims con mayor nivel de carrera presentan tasas de felicidad más altas.

En la etapa de modelado se aplicó una metodología rigurosa:
- **División previa** del dataset (70 % entrenamiento / 30 % prueba, estratificada).
- **One-Hot Encoding** ajustado exclusivamente sobre el conjunto de entrenamiento y aplicado al de prueba sin reajuste, evitando data leakage.
- **Árbol de Decisión** entrenado sobre las variables resultantes.
- **Evaluación sobre ambos conjuntos** para detectar sobreajuste.

El análisis de importancia de variables revela que `moodlets_positivos`, `moodlets_negativos` y `num_amigos` son los factores con mayor peso en las decisiones del modelo, resultado consistente con las hipótesis del EDA y con la lógica del juego: el estado emocional momento a momento y la vida social son los pilares de la felicidad de un Sim.